# OCR Experiments: Setup and Data Collection

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

This notebook sets up our OCR experiment to compare **3 approaches** for extracting text from PAGASA typhoon bulletins:

1. **Surya OCR** — Best AI-native OCR (19.6k stars)
2. **PaddleOCR** — Best traditional deep learning OCR (75k+ stars)
3. **Gemma 4 Vision** — Hypothesis: Vision-first approach (26B model via Ollama)

## Why This Matters

PAGASA bulletins are structured government documents with:
- Storm names, categories, coordinates
- Wind speed tables
- Affected areas and warnings
- Embedded storm track maps

**Key Question:** Can Gemma 4 Vision extract text accurately enough to replace specialized OCR?

## 1. Install Dependencies

In [ ]:
# Core dependencies
!pip install -q pdf2image pillow requests GitPython

# OCR libraries (we'll install these separately in each experiment notebook)
# !pip install surya-ocr
# !pip install paddlepaddle paddleocr

In [ ]:
import os
import sys
from pathlib import Path
from PIL import Image
import requests
from git import Repo
import json
from datetime import datetime

print(f"Python: {sys.version}")
print(f"Working directory: {os.getcwd()}")

## 2. Clone PAGASA Bulletin Archive

We'll use the `pagasa-parser/bulletin-archive` GitHub repository which contains historical typhoon bulletins.

In [ ]:
# Setup directories
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

repo_dir = data_dir / "bulletin-archive"

# Clone repo if not exists
if not repo_dir.exists():
    print("Cloning pagasa-parser/bulletin-archive...")
    Repo.clone_from(
        "https://github.com/pagasa-parser/bulletin-archive.git",
        repo_dir
    )
    print("✓ Clone complete")
else:
    print("✓ Repository already exists")

print(f"\nRepository location: {repo_dir.absolute()}")

## 3. Explore the Data Structure

Let's understand how the bulletins are organized.

In [ ]:
# List storm folders (limited to first 10 for preview)
storm_folders = sorted([d for d in repo_dir.iterdir() if d.is_dir() and not d.name.startswith('.'))][:10]

print(f"Total storm folders found: {len(list(repo_dir.iterdir()))}")
print(f"\nFirst 10 storms:")
for folder in storm_folders:
    pdf_count = len(list(folder.glob("*.pdf")))
    print(f"  {folder.name}: {pdf_count} bulletins")

## 4. Select Sample Bulletins for Testing

We'll select 10 diverse bulletins from different storms and bulletin types (SWB, TCB, TCA).

In [ ]:
# Find all PDFs
all_pdfs = list(repo_dir.glob("**/*.pdf"))
print(f"Total PDFs in archive: {len(all_pdfs)}")

# Select 10 sample PDFs (mix of different storms and bulletin numbers)
# Strategy: Take first storm, last storm, and some from middle
sample_pdfs = []

# Get storms with PDFs
storms_with_pdfs = [d for d in repo_dir.iterdir() if d.is_dir() and list(d.glob("*.pdf"))]

if len(storms_with_pdfs) >= 5:
    # Sample from 5 different storms
    sample_storms = [
        storms_with_pdfs[0],           # First storm
        storms_with_pdfs[len(storms_with_pdfs)//4],   # Quarter way
        storms_with_pdfs[len(storms_with_pdfs)//2],   # Middle
        storms_with_pdfs[3*len(storms_with_pdfs)//4], # Three quarters
        storms_with_pdfs[-1],          # Last storm
    ]
    
    for storm_dir in sample_storms:
        storm_pdfs = list(storm_dir.glob("*.pdf"))
        if storm_pdfs:
            # Take first 2 bulletins from each storm
            sample_pdfs.extend(storm_pdfs[:2])

# Limit to 10 samples
sample_pdfs = sample_pdfs[:10]

print(f"\nSelected {len(sample_pdfs)} sample PDFs:")
for i, pdf in enumerate(sample_pdfs, 1):
    print(f"  {i}. {pdf.parent.name}/{pdf.name}")

## 5. Convert PDFs to Images

For OCR and vision model testing, we need to convert PDF pages to images.

In [ ]:
from pdf2image import convert_from_path

# Create images directory
images_dir = data_dir / "sample_images"
images_dir.mkdir(exist_ok=True)

# Convert PDFs to images
pdf_image_map = {}

for pdf_path in sample_pdfs:
    try:
        # Convert first page only (most bulletins are 1-2 pages)
        images = convert_from_path(pdf_path, first_page=1, last_page=1, dpi=200)
        
        if images:
            # Save image
            image_name = f"{pdf_path.parent.name}_{pdf_path.stem}.png"
            image_path = images_dir / image_name
            images[0].save(image_path, 'PNG')
            
            pdf_image_map[str(pdf_path)] = str(image_path)
            print(f"✓ Converted: {pdf_path.name}")
    except Exception as e:
        print(f"✗ Failed to convert {pdf_path.name}: {e}")

print(f"\n✓ Converted {len(pdf_image_map)} PDFs to images")
print(f"Images saved to: {images_dir.absolute()}")

## 6. Save Sample Metadata

Save the mapping of PDFs to images for use in other notebooks.

In [ ]:
# Save metadata
metadata = {
    "created": datetime.now().isoformat(),
    "total_samples": len(pdf_image_map),
    "samples": [
        {
            "pdf_path": pdf_path,
            "image_path": img_path,
            "storm": Path(pdf_path).parent.name,
            "filename": Path(pdf_path).name
        }
        for pdf_path, img_path in pdf_image_map.items()
    ]
}

metadata_path = data_dir / "sample_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to: {metadata_path.absolute()}")

## 7. Preview Sample Images

Let's display one sample image to verify the conversion quality.

In [ ]:
# Display first sample
if pdf_image_map:
    first_image_path = list(pdf_image_map.values())[0]
    img = Image.open(first_image_path)
    
    print(f"Sample image: {Path(first_image_path).name}")
    print(f"Size: {img.size[0]}x{img.size[1]} pixels")
    
    # Display image
    display(img.resize((800, int(800 * img.size[1] / img.size[0]))))

## 8. Summary and Next Steps

### ✅ Completed
- Cloned PAGASA bulletin archive repository
- Selected 10 diverse sample bulletins
- Converted PDFs to high-resolution images (200 DPI)
- Saved metadata for use in comparison notebooks

### 📝 Next Steps
1. **Notebook 02**: Test Surya OCR on samples
2. **Notebook 03**: Test PaddleOCR on samples  
3. **Notebook 04**: Test Gemma 4 Vision (26B via Ollama)
4. **Notebook 05**: Compare results and decide on production approach

### 📊 Evaluation Criteria
For each approach, we'll manually assess:
- Text extraction accuracy
- Structure preservation (storm name, coordinates, warnings)
- Table handling (wind speeds, affected areas)
- Processing speed
- Integration complexity